In [1]:
import pandas as pd
import numpy as np

csv_path = "data/orders_2024_01.csv"
print(pd.__version__)

3.0.2


In [2]:
df_test = pd.read_csv(csv_path, nrows=5)
print(df_test)
print(df_test.columns.tolist())

   order_id  user_id       order_datetime  product_id     price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.15         1   
1         2     1079  2024-01-03 17:45:43          16   9172.60         1   
2         3     1041  2024-01-06 22:46:38          25   9035.26         2   
3         4     1083  2024-01-08 21:03:51           9  15234.33         3   
4         5     1068  2024-01-03 23:26:53          24   4134.38         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  
['order_id', 'user_id', 'order_datetime', 'product_id', 'price', 'quantity', 'status']


In [3]:
chunk_size = 50_000
total_revenue = 0
rows_count = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    chunk["revenue"] = chunk["price"] * chunk["quantity"]
    total_revenue += chunk["revenue"].sum()
    rows_count += len(chunk)

print("Общее количество строк:", rows_count)
print("Общая выручка:", total_revenue)

Общее количество строк: 5000
Общая выручка: 129378775.5


In [4]:
use_columns = ["order_id", "user_id", "price", "quantity"]

dtypes = {
    "order_id": "int64",
    "user_id": "int32",
    "price": "float32",
    "quantity": "int16",
}

df_opt = pd.read_csv(csv_path, usecols=use_columns, dtype=dtypes)

print(df_opt.info())
print(df_opt.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5000 non-null   int64  
 1   user_id   5000 non-null   int32  
 2   price     5000 non-null   float32
 3   quantity  5000 non-null   int16  
dtypes: float32(1), int16(1), int32(1), int64(1)
memory usage: 88.0 KB
None
   order_id  user_id         price  quantity
0         1     1044  14038.150391         1
1         2     1079   9172.599609         1
2         3     1041   9035.259766         2
3         4     1083  15234.330078         3
4         5     1068   4134.379883         2


In [5]:
df_full = pd.read_csv(csv_path)

mem_full = df_full.memory_usage(deep=True).sum() / 1024**2
mem_opt = df_opt.memory_usage(deep=True).sum() / 1024**2

print(f"Sans optimisation : {mem_full:.2f} MB")
print(f"Avec optimisation : {mem_opt:.2f} MB")
print(f"Économie : {100 * (1 - mem_opt / mem_full):.1f}%")

del df_full  # libère la mémoire

Sans optimisation : 0.54 MB
Avec optimisation : 0.09 MB
Économie : 84.2%


In [6]:
parts = []

for chunk in pd.read_csv(csv_path, usecols=use_columns, dtype=dtypes, chunksize=chunk_size):
    chunk["revenue"] = chunk["price"].astype("float64") * chunk["quantity"]
    parts.append(chunk.groupby("user_id")["revenue"].sum())

user_revenue = (
    pd.concat(parts)
    .groupby(level=0).sum()
    .rename("total_revenue")
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

print(user_revenue.head())

    user_id  total_revenue
80     1081   1.780866e+06
16     1017   1.731081e+06
65     1066   1.721146e+06
88     1089   1.693167e+06
43     1044   1.686822e+06


In [7]:
print(user_revenue["total_revenue"].sum())
print(total_revenue)

129378775.481781
129378775.5


In [8]:
user_revenue.to_csv("user_revenue_2024_01.csv", index=False)